# Meyaar Vision — Florence-2 Experiments (Clean Version)

هدف النوتبوك: تجهيز benchmark بصري بدون data leakage، تشغيل **E0 baseline**، ثم تجارب LoRA بشكل منظم.

**الترتيب:** Data → Injection → Sanity Check → E0 → E1–E5 → Final Test.

> مهم: نقسم الخرائط الأصلية أولًا، ثم نولد الأخطاء داخل كل split. الـTest لا يُستخدم لاختيار أفضل تجربة.

## 0. Setup
شغلي هذه الخلية مرة واحدة في بداية جلسة Colab. لا يوجد حذف/إعادة تثبيت منفصل لـPillow لاحقًا.

In [ ]:
!pip -q install -U \
    "transformers>=5.15.0,<5.16" \
    "peft>=0.17" "accelerate>=1.10" "datasets>=4.0" \
    "scikit-learn>=1.5" "opencv-python-headless>=4.10" \
    "pandas>=2.2" "matplotlib>=3.9" "Pillow==11.3.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Paths + local workspace
نقرأ البيانات من `/content` لتقليل عمليات Google Drive الصغيرة. النتائج المهمة يمكن نسخها للـDrive في النهاية.

In [ ]:
import shutil
from pathlib import Path

DRIVE_PROJECT = Path("/content/drive/MyDrive/Meyaar")
PROJECT_DIR = Path("/content/Meyaar")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

for folder in ["maps", "labels"]:
    src = DRIVE_PROJECT / folder
    dst = PROJECT_DIR / folder
    if not dst.exists():
        print(f"Copying {folder} to local Colab storage...")
        shutil.copytree(src, dst)
    else:
        print(f"{folder} already exists locally ✅")

MAPS_DIR = PROJECT_DIR / "maps"
LABELS_DIR = PROJECT_DIR / "labels"
OUTPUT_DIR = PROJECT_DIR / "vision_experiments"
GENERATED_DIR = OUTPUT_DIR / "generated_dataset"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
RESULTS_CSV = OUTPUT_DIR / "experiment_results.csv"
METADATA_CSV = OUTPUT_DIR / "benchmark_metadata.csv"
SPLIT_CSV = OUTPUT_DIR / "original_map_split.csv"

for p in [OUTPUT_DIR, GENERATED_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Maps:", MAPS_DIR)
print("Labels:", LABELS_DIR)
print("Outputs:", OUTPUT_DIR)

In [ ]:
import os, re, gc, cv2, json, time, random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

import torch
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Read annotations

In [ ]:
ELEMENT_MAP = {
    "title": "title",
    "subtitle": "text",
    "legend-color": "legend",
    "legend-symbol": "legend",
    "legend-mixed": "legend",
    "scale-graphic": "scale",
    "scale-numeric": "scale",
    "orient-arrow": "orientation",
    "mapped_area": "mapped_area",
    "neat_line": "map_frame",
    "neat_line-in": "map_frame",
    "neat_line-out": "map_frame",
    "additional_notes": "text",
    "data_source": "text",
    "credit": "text",
    "inset": "inset",
    "orient-grids": "grid",
}

def annotation_to_bbox(obj):
    pts = obj.get("obj_points", [])
    if not pts:
        return None

    if obj.get("obj_type") == 1:
        p = pts[0]
        if not all(k in p for k in ("x", "y", "w", "h")):
            return None
        x1, y1 = float(p["x"]), float(p["y"])
        x2, y2 = x1 + float(p["w"]), y1 + float(p["h"])
    else:
        valid = [p for p in pts if "x" in p and "y" in p]
        if not valid:
            return None
        xs = [float(p["x"]) for p in valid]
        ys = [float(p["y"]) for p in valid]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)

    return tuple(int(round(v)) for v in (x1, y1, x2, y2))

def clip_bbox(bbox, width, height):
    if bbox is None:
        return None
    x1, y1, x2, y2 = bbox
    x1 = max(0, min(width - 1, x1))
    y1 = max(0, min(height - 1, y1))
    x2 = max(x1 + 1, min(width, x2))
    y2 = max(y1 + 1, min(height, y2))
    return (x1, y1, x2, y2)

def image_name_from_json(json_path):
    return json_path.name[:-5] if json_path.name.endswith(".json") else json_path.stem

def find_image(image_name):
    direct = MAPS_DIR / image_name
    if direct.exists():
        return direct
    matches = list(MAPS_DIR.rglob(image_name))
    return matches[0] if matches else None

def load_annotation_file(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

json_files = sorted(LABELS_DIR.rglob("*.json"))
print("JSON files:", len(json_files))

raw_counts = Counter()
obj_type_counts = Counter()
paired = []

for jp in json_files:
    anns = load_annotation_file(jp)

    for obj in anns:
        raw_counts[obj.get("f_name", "UNKNOWN")] += 1
        obj_type_counts[obj.get("obj_type", "UNKNOWN")] += 1

    image_name = image_name_from_json(jp)
    img_path = find_image(image_name)
    if img_path is not None:
        paired.append((img_path, jp))

print("\nAnnotation classes:")
display(pd.DataFrame(raw_counts.most_common(), columns=["raw_label", "count"]))

print("\nObject types:")
display(pd.DataFrame(obj_type_counts.items(), columns=["obj_type", "count"]))

print(f"\nMatched image/JSON pairs: {len(paired)} / {len(json_files)}")

## 3. Build original-map index

In [ ]:
records = []

for img_path, json_path in paired:
    try:
        with Image.open(img_path) as im:
            width, height = im.size
    except Exception as e:
        print("Skipping unreadable image:", img_path, e)
        continue

    anns = load_annotation_file(json_path)
    elements = defaultdict(list)

    for obj in anns:
        raw_name = obj.get("f_name")
        merged = ELEMENT_MAP.get(raw_name)
        bbox = clip_bbox(annotation_to_bbox(obj), width, height)

        if merged and bbox:
            elements[merged].append({
                "raw_name": raw_name,
                "bbox": bbox,
                "obj_type": obj.get("obj_type")
            })

    records.append({
        "map_id": img_path.name,
        "image_path": str(img_path),
        "json_path": str(json_path),
        "width": width,
        "height": height,
        "elements": dict(elements),
        "has_title": int(bool(elements.get("title"))),
        "has_legend": int(bool(elements.get("legend"))),
        "has_scale": int(bool(elements.get("scale"))),
        "has_orientation": int(bool(elements.get("orientation"))),
        "has_text": int(len(elements.get("text", [])) + len(elements.get("title", [])) >= 2),
    })

original_df = pd.DataFrame(records)

print("Original maps:", len(original_df))
display(original_df.head())

presence_cols = ["has_title", "has_legend", "has_scale", "has_orientation", "has_text"]
display(original_df[presence_cols].mean().rename("fraction_of_maps").to_frame())

## 4. Split originals before injection

In [ ]:
rng = np.random.default_rng(SEED)
indices = np.arange(len(original_df))
rng.shuffle(indices)

n = len(indices)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train+n_val]
test_idx = indices[n_train+n_val:]

original_df["split"] = ""
original_df.loc[train_idx, "split"] = "train"
original_df.loc[val_idx, "split"] = "val"
original_df.loc[test_idx, "split"] = "test"

print(original_df["split"].value_counts())
display(original_df.groupby("split")[presence_cols].mean().round(3))

original_df.drop(columns=["elements"]).to_csv(SPLIT_CSV, index=False)
print("Saved:", SPLIT_CSV)

## 5. Controlled visual error injection

- **Stage 1:** missing title / legend / scale / orientation
- **Stage 2:** overlap / clipping
- **Stage 3:** text overlap / illegible text

ابدئي بـStage 1 فقط.

In [ ]:
ERROR_LABELS = [
    "clean",
    "missing_title",
    "missing_legend",
    "missing_scale",
    "missing_orientation",
    "element_overlap",
    "element_clipping",
    "text_overlap",
    "illegible_text",
]

def bbox_area(b):
    x1, y1, x2, y2 = b
    return max(0, x2-x1) * max(0, y2-y1)

def expand_bbox(b, pad, w, h):
    x1, y1, x2, y2 = b
    return (
        max(0, x1-pad), max(0, y1-pad),
        min(w, x2+pad), min(h, y2+pad)
    )

def inpaint_region(pil_img, bbox, pad=3):
    arr = np.array(pil_img.convert("RGB"))
    h, w = arr.shape[:2]
    x1, y1, x2, y2 = expand_bbox(bbox, pad, w, h)

    mask = np.zeros((h, w), dtype=np.uint8)
    mask[y1:y2, x1:x2] = 255

    bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    result = cv2.inpaint(bgr, mask, 5, cv2.INPAINT_TELEA)
    return Image.fromarray(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))

def crop_element(img, bbox):
    return img.crop(bbox)

def paste_overlap(img, source_bbox, target_bbox):
    out = img.copy()
    patch = crop_element(img, source_bbox)
    pw, ph = patch.size

    tx1, ty1, tx2, ty2 = target_bbox
    cx = (tx1 + tx2) // 2
    cy = (ty1 + ty2) // 2

    nx = int(cx - pw * 0.5)
    ny = int(cy - ph * 0.5)

    nx = max(0, min(img.width - 1, nx))
    ny = max(0, min(img.height - 1, ny))

    out.paste(patch, (nx, ny))
    pasted = (nx, ny, min(img.width, nx+pw), min(img.height, ny+ph))
    return out, pasted

def paste_clipped(img, source_bbox):
    out = img.copy()
    patch = crop_element(img, source_bbox)
    pw, ph = patch.size
    edge = random.choice(["left", "right", "top", "bottom"])

    if edge == "left":
        nx, ny = -pw//2, max(0, min(img.height-ph, source_bbox[1]))
    elif edge == "right":
        nx, ny = img.width-pw//2, max(0, min(img.height-ph, source_bbox[1]))
    elif edge == "top":
        nx, ny = max(0, min(img.width-pw, source_bbox[0])), -ph//2
    else:
        nx, ny = max(0, min(img.width-pw, source_bbox[0])), img.height-ph//2

    out.paste(patch, (nx, ny))
    visible = (
        max(0, nx), max(0, ny),
        min(img.width, nx+pw), min(img.height, ny+ph)
    )
    return out, visible

def make_text_overlap(img, text_bboxes):
    if len(text_bboxes) < 2:
        return None, None
    b1, b2 = random.sample(text_bboxes, 2)
    if bbox_area(b1) < 20 or bbox_area(b2) < 20:
        return None, None
    return paste_overlap(img, b1, b2)

def make_illegible(img, bbox):
    out = img.copy()
    patch = crop_element(img, bbox)

    small_w = max(2, patch.width // 4)
    small_h = max(2, patch.height // 4)

    patch = patch.resize((small_w, small_h), Image.Resampling.BILINEAR)
    patch = patch.resize(
        (max(2, bbox[2]-bbox[0]), max(2, bbox[3]-bbox[1])),
        Image.Resampling.NEAREST
    )
    patch = patch.filter(ImageFilter.GaussianBlur(radius=2.0))
    out.paste(patch, (bbox[0], bbox[1]))
    return out

def save_generated(img, split, map_id, error_type):
    stem = Path(map_id).stem
    folder = GENERATED_DIR / split / error_type
    folder.mkdir(parents=True, exist_ok=True)

    path = folder / f"{stem}__{error_type}.jpg"
    img.convert("RGB").save(path, quality=92)
    return str(path)

In [ ]:
MAX_PER_ERROR = 100
INCLUDE_STAGES = {1}
REGENERATE_DATASET = False  # غيّريها True فقط إذا تبين إعادة الحقن من الصفر

MISSING_CONFIG = {
    "missing_title": "title",
    "missing_legend": "legend",
    "missing_scale": "scale",
    "missing_orientation": "orientation",
}

def can_make_error(row, error_type):
    elements = row["elements"]

    if error_type in MISSING_CONFIG:
        return bool(elements.get(MISSING_CONFIG[error_type]))
    if error_type == "element_overlap":
        boxes = [x["bbox"] for k, vals in elements.items() if k != "mapped_area" for x in vals]
        return len(boxes) >= 2
    if error_type == "element_clipping":
        boxes = [x["bbox"] for k in ["legend", "scale", "title", "orientation", "text", "inset"] for x in elements.get(k, [])]
        return len(boxes) >= 1
    if error_type == "text_overlap":
        boxes = [x["bbox"] for x in elements.get("title", [])] + [x["bbox"] for x in elements.get("text", [])]
        return len(boxes) >= 2
    if error_type == "illegible_text":
        boxes = [x["bbox"] for x in elements.get("title", [])] + [x["bbox"] for x in elements.get("text", [])]
        return len(boxes) >= 1
    return False

In [ ]:
def generate_benchmark(original_df, overwrite=False):
    seed_everything(SEED)

    if overwrite and GENERATED_DIR.exists():
        shutil.rmtree(GENERATED_DIR)
        GENERATED_DIR.mkdir(parents=True, exist_ok=True)

    rows = []

    # Add clean samples
    for _, row in original_df.iterrows():
        rows.append({
            "map_id": row["map_id"],
            "split": row["split"],
            "error_type": "clean",
            "affected_element": "",
            "error_bbox": "",
            "image_path": row["image_path"],
            "is_injected": 0,
        })

    stage_errors = []

    if 1 in INCLUDE_STAGES:
        stage_errors += list(MISSING_CONFIG.keys())

    if 2 in INCLUDE_STAGES:
        stage_errors += [
            "element_overlap",
            "element_clipping"
        ]

    if 3 in INCLUDE_STAGES:
        stage_errors += [
            "text_overlap",
            "illegible_text"
        ]

    for split in ["train", "val", "test"]:

        split_df = original_df[
            original_df["split"] == split
        ].copy()

        for error_type in stage_errors:

            candidates = split_df[
                split_df.apply(
                    lambda r: can_make_error(r, error_type),
                    axis=1
                )
            ].copy()

            candidates = candidates.sample(
                frac=1,
                random_state=SEED
            )

            if MAX_PER_ERROR is not None:
                candidates = candidates.head(MAX_PER_ERROR)

            for _, row in candidates.iterrows():

                try:
                    img = Image.open(
                        row["image_path"]
                    ).convert("RGB")

                    elements = row["elements"]

                    affected = ""
                    error_bbox = None


                    # -------------------------
                    # Missing Element
                    # -------------------------
                    if error_type in MISSING_CONFIG:

                        affected = MISSING_CONFIG[error_type]

                        # Get ALL boxes for this element
                        boxes = [
                            item["bbox"]
                            for item in elements[affected]
                        ]

                        corrupted = img.copy()

                        # Remove ALL instances
                        for bbox in boxes:
                            corrupted = inpaint_region(
                                corrupted,
                                bbox
                            )

                        error_bbox = boxes


                    # -------------------------
                    # Element Overlap
                    # -------------------------
                    elif error_type == "element_overlap":

                        items = []

                        for k, vals in elements.items():

                            if k != "mapped_area":

                                for v in vals:
                                    items.append(
                                        (k, v["bbox"])
                                    )

                        (
                            (src_name, src_bbox),
                            (tgt_name, tgt_bbox)
                        ) = random.sample(items, 2)

                        corrupted, error_bbox = paste_overlap(
                            img,
                            src_bbox,
                            tgt_bbox
                        )

                        affected = (
                            f"{src_name}+{tgt_name}"
                        )


                    # -------------------------
                    # Element Clipping
                    # -------------------------
                    elif error_type == "element_clipping":

                        items = []

                        for k in [
                            "legend",
                            "scale",
                            "title",
                            "orientation",
                            "text",
                            "inset"
                        ]:

                            for v in elements.get(k, []):
                                items.append(
                                    (k, v["bbox"])
                                )

                        affected, src_bbox = random.choice(
                            items
                        )

                        corrupted, error_bbox = paste_clipped(
                            img,
                            src_bbox
                        )


                    # -------------------------
                    # Text Overlap
                    # -------------------------
                    elif error_type == "text_overlap":

                        text_boxes = [
                            x["bbox"]
                            for x in elements.get(
                                "title",
                                []
                            )
                        ]

                        text_boxes += [
                            x["bbox"]
                            for x in elements.get(
                                "text",
                                []
                            )
                        ]

                        corrupted, error_bbox = (
                            make_text_overlap(
                                img,
                                text_boxes
                            )
                        )

                        if corrupted is None:
                            continue

                        affected = "text"


                    # -------------------------
                    # Illegible Text
                    # -------------------------
                    elif error_type == "illegible_text":

                        text_boxes = [
                            x["bbox"]
                            for x in elements.get(
                                "title",
                                []
                            )
                        ]

                        text_boxes += [
                            x["bbox"]
                            for x in elements.get(
                                "text",
                                []
                            )
                        ]

                        bbox = random.choice(
                            text_boxes
                        )

                        corrupted = make_illegible(
                            img,
                            bbox
                        )

                        error_bbox = bbox
                        affected = "text"


                    # -------------------------
                    # Save generated image
                    # -------------------------
                    out_path = save_generated(
                        corrupted,
                        split,
                        row["map_id"],
                        error_type
                    )

                    rows.append({
                        "map_id": row["map_id"],
                        "split": split,
                        "error_type": error_type,
                        "affected_element": affected,
                        "error_bbox": json.dumps(error_bbox),
                        "image_path": out_path,
                        "is_injected": 1,
                    })

                except Exception as e:

                    print(
                        "Generation failed:",
                        row["map_id"],
                        error_type,
                        e
                    )

    meta = pd.DataFrame(rows)

    meta.to_csv(
        METADATA_CSV,
        index=False
    )

    return meta


In [ ]:
if METADATA_CSV.exists() and not REGENERATE_DATASET:
    benchmark_df = pd.read_csv(METADATA_CSV)
    print("Loaded existing benchmark metadata ✅")
else:
    benchmark_df = generate_benchmark(original_df, overwrite=REGENERATE_DATASET)
    print("Generated benchmark ✅")

print("Benchmark samples:", len(benchmark_df))
display(pd.crosstab(benchmark_df["error_type"], benchmark_df["split"]))
print("Metadata:", METADATA_CSV)

## 6. Visual sanity check
راجعي العينات قبل أي تدريب.

In [ ]:
def compare_original_corrupted(df, error_type, n=3):

    # نختار فقط الصور المحقونة بهذا الخطأ
    filtered = df[
        (df["error_type"] == error_type) &
        (df["is_injected"] == 1)
    ].copy()

    if len(filtered) == 0:
        print(f"No injected samples found for: {error_type}")
        return

    # اختيار عينات ثابتة
    samples = filtered.sample(
        n=min(n, len(filtered)),
        random_state=42
    )

    for _, row in samples.iterrows():

        # نجيب مسار الصورة الأصلية من original_df
        original_match = original_df[
            original_df["map_id"] == row["map_id"]
        ]

        if len(original_match) == 0:
            print("Original image not found:", row["map_id"])
            continue

        original_path = original_match["image_path"].iloc[0]

        # فتح الصور
        original = Image.open(
            original_path
        ).convert("RGB")

        corrupted = Image.open(
            row["image_path"]
        ).convert("RGB")

        # عرض Original و Corrupted جنب بعض
        plt.figure(figsize=(14, 7))

        plt.subplot(1, 2, 1)
        plt.imshow(original)
        plt.title(
            f"Original | {row['map_id']}"
        )
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(corrupted)
        plt.title(
            f"{error_type} | affected={row['affected_element']}"
        )
        plt.axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
for error_name in [
    "missing_legend",
    "missing_scale",
    "missing_title",
    "missing_orientation",
]:
    compare_original_corrupted(benchmark_df, error_name, n=3)

## 7. Experiment scopes

In [ ]:
STAGE1_LABELS = [
    "clean",
    "missing_title",
    "missing_legend",
    "missing_scale",
    "missing_orientation",
]

STAGE2_LABELS = STAGE1_LABELS + [
    "element_overlap",
    "element_clipping",
]

FULL_LABELS = STAGE2_LABELS + [
    "text_overlap",
    "illegible_text",
]

def subset_for_labels(df, labels):
    return df[df["error_type"].isin(labels)].reset_index(drop=True)

stage1_df = subset_for_labels(benchmark_df, STAGE1_LABELS)
stage2_df = subset_for_labels(benchmark_df, STAGE2_LABELS)
full_df = subset_for_labels(benchmark_df, FULL_LABELS)

print("Stage 1:", len(stage1_df))
print("Stage 2:", len(stage2_df))
print("Full:", len(full_df))

## 8. Florence-2 model
نستخدم النسخة المتوافقة مع Transformers: `florence-community/Florence-2-base-ft`.

على CPU ينفع للاختبارات الصغيرة، لكن LoRA يُفضّل GPU.

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "florence-community/Florence-2-base-ft"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
).to(DEVICE)
model.eval()

print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("Florence-2 loaded successfully ✅")

## 9. Classification prompt + constrained baseline
Florence-2 قد يرجع نصًا حرًا مثل `missing` بدل label كامل. لذلك E0 يقارن loss لكل label مسموح ويختار الأقل. هذا يعطي baseline قابل للتقييم بدون fine-tuning.

In [ ]:
STAGE1_PROMPT = """
Classify the visual map quality issue in this image.
Return exactly ONE label from:
clean
missing_title
missing_legend
missing_scale
missing_orientation
Do not return any explanation.
""".strip()

FULL_PROMPT = (
    "Classify the visual map quality issue. Return exactly one label from: "
    "clean, missing_title, missing_legend, missing_scale, missing_orientation, "
    "element_overlap, element_clipping, text_overlap, illegible_text."
)

def prompt_for_labels(allowed_labels):
    if set(allowed_labels) == set(STAGE1_LABELS):
        return STAGE1_PROMPT
    return "Classify the visual map quality issue. Return exactly one label from: " + ", ".join(allowed_labels)

def model_device(model_obj):
    return next(model_obj.parameters()).device

def predict_one(image_path, allowed_labels, model_obj=None):
    model_obj = model if model_obj is None else model_obj
    device = model_device(model_obj)
    image = Image.open(image_path).convert("RGB")
    prompt = prompt_for_labels(allowed_labels)

    inputs = processor(text=prompt, images=image, return_tensors="pt")
    inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

    scores = {}
    for label in allowed_labels:
        label_ids = processor.tokenizer(label, return_tensors="pt").input_ids.to(device)
        with torch.inference_mode():
            outputs = model_obj(**inputs, labels=label_ids)
        scores[label] = float(outputs.loss.detach().cpu())

    prediction = min(scores, key=scores.get)
    return prediction, scores

### Optional smoke test
على CPU قد يأخذ وقتًا. شغليه على صورة واحدة فقط للتأكد أن الـpipeline يعمل.

In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    sample_row = benchmark_df.iloc[0]
    pred, scores = predict_one(sample_row["image_path"], STAGE1_LABELS)
    print("True label:", sample_row["error_type"])
    print("Prediction:", pred)
    print("Scores:")
    for label, score in sorted(scores.items(), key=lambda x: x[1]):
        print(f"  {label:22s} {score:.4f}")

## 10. Evaluation utilities

In [ ]:
def evaluate_model(eval_df, allowed_labels, model_obj=None, max_samples=None, verbose=False):
    df = eval_df.copy()
    if max_samples is not None and len(df) > max_samples:
        df = df.sample(max_samples, random_state=SEED).reset_index(drop=True)

    y_true, y_pred, raw_outputs = [], [], []
    start = time.time()

    for i, row in df.iterrows():
        pred, scores = predict_one(
            row["image_path"],
            allowed_labels=allowed_labels,
            model_obj=model_obj,
        )
        y_true.append(row["error_type"])
        y_pred.append(pred)
        raw_outputs.append(scores)

        if verbose and i < 5:
            print("TRUE:", row["error_type"], "| PRED:", pred, "| SCORES:", scores)

    elapsed = time.time() - start
    accuracy = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=allowed_labels, average="macro", zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": p,
        "recall_macro": r,
        "f1_macro": f1,
        "seconds": elapsed,
        "n_samples": len(y_true),
        "y_true": y_true,
        "y_pred": y_pred,
        "raw_outputs": raw_outputs,
    }

def show_eval_details(result, labels):
    print(classification_report(result["y_true"], result["y_pred"], labels=labels, zero_division=0))
    cm = confusion_matrix(result["y_true"], result["y_pred"], labels=labels)
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(11, 9))
    disp.plot(ax=ax, xticks_rotation=45, values_format="d")
    plt.title("Confusion Matrix")
    plt.show()

## 11. Experiment tracker

In [ ]:
RESULT_COLUMNS = [
    "experiment",
    "model",
    "scope",
    "training_method",
    "split",
    "n_samples",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "seconds",
    "trainable_params",
    "notes",
]

def load_results():
    if RESULTS_CSV.exists():
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame(columns=RESULT_COLUMNS)

def add_result(
    experiment,
    result,
    scope,
    training_method,
    split="val",
    trainable_params=0,
    notes=""
):
    results = load_results()

    row = {
        "experiment": experiment,
        "model": MODEL_ID,
        "scope": scope,
        "training_method": training_method,
        "split": split,
        "n_samples": result["n_samples"],
        "accuracy": result["accuracy"],
        "precision_macro": result["precision_macro"],
        "recall_macro": result["recall_macro"],
        "f1_macro": result["f1_macro"],
        "seconds": result["seconds"],
        "trainable_params": trainable_params,
        "notes": notes,
    }

    mask = (
        (results["experiment"] == experiment) &
        (results["split"] == split)
    )

    results = results.loc[~mask]
    results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)
    results.to_csv(RESULTS_CSV, index=False)
    return results

# E0 — Zero-shot Baseline
ابدئي بعينة صغيرة. للمقارنة النهائية مع E1 استخدمي نفس validation subset أو شغلي E0 على الـvalidation كامل.

In [ ]:
RUN_E0 = False
BASELINE_MAX_SAMPLES = 20 if DEVICE == "cpu" else 100

if RUN_E0:
    val_stage1 = stage1_df[stage1_df["split"] == "val"].reset_index(drop=True)
    baseline_result = evaluate_model(
        val_stage1,
        allowed_labels=STAGE1_LABELS,
        model_obj=model,
        max_samples=BASELINE_MAX_SAMPLES,
        verbose=True,
    )

    print({k: baseline_result[k] for k in ["accuracy", "precision_macro", "recall_macro", "f1_macro", "seconds", "n_samples"]})
    show_eval_details(baseline_result, STAGE1_LABELS)

    results_df = add_result(
        experiment="E0_zero_shot",
        result=baseline_result,
        scope="stage1_missing_elements",
        training_method="constrained zero-shot scoring",
        trainable_params=0,
        notes=f"max_samples={BASELINE_MAX_SAMPLES}",
    )
    display(results_df)

## 12. LoRA setup
شغلي التدريب فقط عند توفر GPU.

In [ ]:
from torch.utils.data import Dataset
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import TrainingArguments, Trainer

class MeyaarVisionDataset(Dataset):
    def __init__(self, df, allowed_labels):
        self.df = df.reset_index(drop=True)
        self.allowed_labels = allowed_labels
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        return {
            "image_path": r["image_path"],
            "target_text": r["error_type"],
            "prompt": prompt_for_labels(self.allowed_labels),
        }

class FlorenceCollator:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, batch):
        images = [Image.open(x["image_path"]).convert("RGB") for x in batch]
        prompts = [x["prompt"] for x in batch]
        targets = [x["target_text"] for x in batch]

        inputs = self.processor(text=prompts, images=images, return_tensors="pt", padding=True)
        target_tokens = self.processor.tokenizer(targets, return_tensors="pt", padding=True)
        labels = target_tokens["input_ids"]
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100
        inputs["labels"] = labels
        return inputs

collator = FlorenceCollator(processor)

def fresh_base_model():
    return AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
    )

def make_lora_model(r=8, alpha=16, dropout=0.05):
    if not torch.cuda.is_available():
        raise RuntimeError("LoRA training is intentionally disabled on CPU. Use a GPU runtime.")

    base = fresh_base_model()
    matched = [name for name, _ in base.named_modules() if name.endswith("q_proj") or name.endswith("v_proj")]
    print("Matched q_proj/v_proj modules:", len(matched))
    print(matched[:20])
    if not matched:
        raise RuntimeError("No q_proj/v_proj modules found; inspect model.named_modules().")

    config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
        target_modules=["q_proj", "v_proj"],
    )
    m = get_peft_model(base, config)
    m.print_trainable_parameters()
    return m

def count_trainable_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 13. Training helper

In [ ]:
def train_lora_experiment(
    experiment_name, train_df, val_df, allowed_labels,
    epochs=3, lr=2e-4, r=8, alpha=16, dropout=0.05,
    batch_size=2, grad_accum=8,
):
    seed_everything(SEED)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    lora_model = make_lora_model(r=r, alpha=alpha, dropout=dropout)
    trainable = count_trainable_params(lora_model)

    out_dir = CHECKPOINT_DIR / experiment_name
    out_dir.mkdir(parents=True, exist_ok=True)

    args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        fp16=torch.cuda.is_available(),
        logging_steps=20,
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=2,
        seed=SEED,
    )

    trainer = Trainer(
        model=lora_model,
        args=args,
        train_dataset=MeyaarVisionDataset(train_df, allowed_labels),
        data_collator=collator,
    )

    start = time.time()
    trainer.train()
    train_seconds = time.time() - start

    adapter_dir = out_dir / "final_adapter"
    trainer.model.save_pretrained(adapter_dir)
    processor.save_pretrained(adapter_dir)

    lora_model.eval()
    result = evaluate_model(val_df, allowed_labels, model_obj=lora_model, max_samples=None)
    result["seconds"] += train_seconds
    return lora_model, result, trainable, str(adapter_dir)

# E1 — LoRA on Stage 1

Question:

**Does Meyaar-specific fine-tuning improve missing-element detection over the zero-shot baseline?**

In [ ]:
RUN_E1 = False

if RUN_E1:
    train_e1 = stage1_df[
        stage1_df["split"] == "train"
    ].reset_index(drop=True)

    val_e1 = stage1_df[
        stage1_df["split"] == "val"
    ].reset_index(drop=True)

    model_e1, e1_result, e1_trainable, e1_adapter = train_lora_experiment(
        experiment_name="E1_lora_stage1",
        train_df=train_e1,
        val_df=val_e1,
        allowed_labels=STAGE1_LABELS,
        epochs=3,
        lr=2e-4,
        r=8
    )

    results_df = add_result(
        experiment="E1_lora_stage1",
        result=e1_result,
        scope="stage1_missing_elements",
        training_method="LoRA r=8",
        trainable_params=e1_trainable,
        notes=f"adapter={e1_adapter}"
    )

    display(results_df)
    show_eval_details(e1_result, STAGE1_LABELS)

# E2 — Add Overlap + Clipping

Question:

**What happens when we expand the task from missing elements to layout errors?**

In [ ]:
RUN_E2 = False

if RUN_E2:
    train_e2 = stage2_df[
        stage2_df["split"] == "train"
    ].reset_index(drop=True)

    val_e2 = stage2_df[
        stage2_df["split"] == "val"
    ].reset_index(drop=True)

    model_e2, e2_result, e2_trainable, e2_adapter = train_lora_experiment(
        experiment_name="E2_lora_stage2",
        train_df=train_e2,
        val_df=val_e2,
        allowed_labels=STAGE2_LABELS,
        epochs=3,
        lr=2e-4,
        r=8
    )

    results_df = add_result(
        experiment="E2_lora_stage2",
        result=e2_result,
        scope="stage1_plus_overlap_clipping",
        training_method="LoRA r=8",
        trainable_params=e2_trainable,
        notes=f"adapter={e2_adapter}"
    )

    display(results_df)
    show_eval_details(e2_result, STAGE2_LABELS)

# E3 — Full Vision Scope

Adds:
- Text/Label Overlap
- Illegible Text

In [ ]:
RUN_E3 = False

if RUN_E3:
    train_e3 = full_df[
        full_df["split"] == "train"
    ].reset_index(drop=True)

    val_e3 = full_df[
        full_df["split"] == "val"
    ].reset_index(drop=True)

    model_e3, e3_result, e3_trainable, e3_adapter = train_lora_experiment(
        experiment_name="E3_lora_full",
        train_df=train_e3,
        val_df=val_e3,
        allowed_labels=FULL_LABELS,
        epochs=3,
        lr=2e-4,
        r=8
    )

    results_df = add_result(
        experiment="E3_lora_full",
        result=e3_result,
        scope="full_vision_scope",
        training_method="LoRA r=8",
        trainable_params=e3_trainable,
        notes=f"adapter={e3_adapter}"
    )

    display(results_df)
    show_eval_details(e3_result, FULL_LABELS)

# E4 — Class Balancing

Oversampling is applied to Train only.
Validation/Test stay unchanged.

In [ ]:
def oversample_train(df, label_col="error_type", seed=SEED):
    counts = df[label_col].value_counts()
    max_count = counts.max()

    balanced = []

    for label, group in df.groupby(label_col):
        balanced.append(
            group.sample(
                max_count,
                replace=True,
                random_state=seed
            )
        )

    out = pd.concat(balanced, ignore_index=True)
    return out.sample(frac=1, random_state=seed).reset_index(drop=True)

RUN_E4 = False

if RUN_E4:
    train_raw = full_df[
        full_df["split"] == "train"
    ].reset_index(drop=True)

    train_e4 = oversample_train(train_raw)

    val_e4 = full_df[
        full_df["split"] == "val"
    ].reset_index(drop=True)

    print("Before:")
    display(train_raw["error_type"].value_counts().to_frame("count"))

    print("After:")
    display(train_e4["error_type"].value_counts().to_frame("count"))

    model_e4, e4_result, e4_trainable, e4_adapter = train_lora_experiment(
        experiment_name="E4_lora_balanced",
        train_df=train_e4,
        val_df=val_e4,
        allowed_labels=FULL_LABELS,
        epochs=3,
        lr=2e-4,
        r=8
    )

    results_df = add_result(
        experiment="E4_lora_balanced",
        result=e4_result,
        scope="full_vision_scope",
        training_method="LoRA r=8 + oversampling",
        trainable_params=e4_trainable,
        notes=f"adapter={e4_adapter}"
    )

    display(results_df)
    show_eval_details(e4_result, FULL_LABELS)

# E5 — LoRA Capacity

Compare `r=8` with a larger `r=16` adapter.

In [ ]:
RUN_E5 = False

if RUN_E5:
    train_e5 = full_df[
        full_df["split"] == "train"
    ].reset_index(drop=True)

    val_e5 = full_df[
        full_df["split"] == "val"
    ].reset_index(drop=True)

    model_e5, e5_result, e5_trainable, e5_adapter = train_lora_experiment(
        experiment_name="E5_lora_r16",
        train_df=train_e5,
        val_df=val_e5,
        allowed_labels=FULL_LABELS,
        epochs=3,
        lr=2e-4,
        r=16,
        alpha=32
    )

    results_df = add_result(
        experiment="E5_lora_r16",
        result=e5_result,
        scope="full_vision_scope",
        training_method="LoRA r=16",
        trainable_params=e5_trainable,
        notes=f"adapter={e5_adapter}"
    )

    display(results_df)
    show_eval_details(e5_result, FULL_LABELS)

## 14. Compare experiments

In [ ]:
results_df = load_results()
if len(results_df):
    display(results_df.sort_values(["split", "f1_macro"], ascending=[True, False]))
else:
    print("No saved experiment results yet.")

## 15. Load saved LoRA adapter

In [ ]:
def load_saved_adapter(adapter_path):
    base = fresh_base_model()
    loaded = PeftModel.from_pretrained(base, adapter_path)
    loaded = loaded.to(DEVICE)
    loaded.eval()
    return loaded

# 16. Final Test

Run only after selecting the best experiment from Validation.

In [ ]:
RUN_FINAL_TEST = False

BEST_EXPERIMENT = "E4_lora_balanced"
BEST_ADAPTER_PATH = (
    CHECKPOINT_DIR /
    BEST_EXPERIMENT /
    "final_adapter"
)

if RUN_FINAL_TEST:
    best_model = load_saved_adapter(
        str(BEST_ADAPTER_PATH)
    )

    test_df = full_df[
        full_df["split"] == "test"
    ].reset_index(drop=True)

    final_result = evaluate_model(
        test_df,
        allowed_labels=FULL_LABELS,
        model_obj=best_model,
        max_samples=None,
        verbose=True
    )

    results_df = add_result(
        experiment=BEST_EXPERIMENT,
        result=final_result,
        scope="full_vision_scope",
        training_method="selected_best_adapter",
        split="test",
        trainable_params=0,
        notes="Final untouched test evaluation"
    )

    display(results_df)
    show_eval_details(
        final_result,
        FULL_LABELS
    )

## 17. Group missing-element subtypes

The project documentation has a broader **Missing Required Map Element** category.

This optional view merges:
- missing title
- missing legend
- missing scale
- missing orientation

into one group for reporting.

In [ ]:
def aggregate_missing_family(labels):
    missing = {
        "missing_title",
        "missing_legend",
        "missing_scale",
        "missing_orientation",
    }

    return [
        "missing_required_map_element"
        if x in missing else x
        for x in labels
    ]

# Example after final_result exists:
# y_true_grouped = aggregate_missing_family(final_result["y_true"])
# y_pred_grouped = aggregate_missing_family(final_result["y_pred"])
# print(classification_report(y_true_grouped, y_pred_grouped, zero_division=0))

## 18. IoU helper for a later localization experiment

The current experiments classify the error type.

The benchmark still stores `error_bbox`, so Florence-2 localization can be added later and evaluated with IoU.

In [ ]:
def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    inter = max(0, ix2-ix1) * max(0, iy2-iy1)

    area_a = max(0, ax2-ax1) * max(0, ay2-ay1)
    area_b = max(0, bx2-bx1) * max(0, by2-by1)

    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

## 19. Experiment plan

| ID | Experiment | Change |
|---|---|---|
| E0 | Zero-shot baseline | No Meyaar training |
| E1 | LoRA Stage 1 | Missing required elements |
| E2 | LoRA Stage 1+2 | + overlap and clipping |
| E3 | LoRA Full | + text overlap and illegible text |
| E4 | LoRA Full + balancing | Tests class imbalance |
| E5 | LoRA r=16 | Tests adapter capacity |
| Final | Best model on Test | One final unbiased evaluation |

### Metrics
- Accuracy
- Macro Precision
- Macro Recall
- Macro F1
- Per-class F1
- Confusion Matrix

## 20. Reproducibility checklist

- [ ] `SEED = 42`
- [ ] Original maps split before injection
- [ ] No map ID appears in multiple splits
- [ ] Validation used to choose experiments
- [ ] Test used only for final evaluation
- [ ] Injected examples visually inspected
- [ ] Class counts reported
- [ ] Results saved to one CSV

In [ ]:
leak = (
    benchmark_df.groupby("map_id")["split"]
    .nunique()
    .sort_values(ascending=False)
)

assert leak.max() == 1, (
    "Data leakage detected: "
    "one original map appears in multiple splits!"
)

print("Leakage check passed ✅")

print("\nFinal class distribution:")
display(
    pd.crosstab(
        benchmark_df["error_type"],
        benchmark_df["split"]
    )
)

print("\nExperiment results file:")
print(RESULTS_CSV)

## 21. Save important outputs back to Google Drive
يشمل metadata/results/splits/checkpoints، ولا ينسخ generated images افتراضيًا.

In [ ]:
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    drive_out = DRIVE_PROJECT / "vision_experiments"
    drive_out.mkdir(parents=True, exist_ok=True)

    for file_path in [METADATA_CSV, RESULTS_CSV, SPLIT_CSV]:
        if Path(file_path).exists():
            shutil.copy2(file_path, drive_out / Path(file_path).name)

    if CHECKPOINT_DIR.exists():
        dst = drive_out / "checkpoints"
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(CHECKPOINT_DIR, dst)

    print("Important outputs copied to Drive ✅")